In [2]:
import pandas as pd
import os
import ast


# Inspect retrieval scores

In [ ]:
# LOAD RESULTS 

run_id = '56fac2506b5b498f8d9509b4dc8109ce'
path_to_results = os.path.join('../../',
            'output', run_id)

results = pd.read_csv(os.path.join(path_to_results, "03_co2_emission_table2_w_query_responses.csv"),
                dtype={
                    'extracted_scope_from_llm_orig': str,
                    'extracted_scope_from_llm': str,
                    'page_number_used_by_llm': str,
                    'page_number_to_llm': str
            })
results['page_numbers_tried_by_llm'] = results['page_numbers_tried_by_llm'].apply(ast.literal_eval)

results_matched = pd.read_csv(os.path.join(path_to_results, "04a_results_available_in_report.csv"),
                dtype={
                    'extracted_scope_from_llm_orig': str,
                    'extracted_scope_from_llm': str,
                    'page_number_used_by_llm': str,
                    'page_number_to_llm': str
            })
results_matched['page_numbers_tried_by_llm'] = results_matched['page_numbers_tried_by_llm'].apply(ast.literal_eval)

In [4]:
results.columns

Index(['extracted_year_from_llm_orig', 'extracted_scope_from_llm_orig',
       'extracted_value_from_llm_orig', 'extracted_unit_from_llm',
       'value_probability', 'unit_probability', 'extracted_scope_from_llm',
       'extracted_year_from_llm', 'extracted_value_from_llm',
       'raw_llm_response', 'page_number_used_by_llm', 'report_name',
       'normalized_unit_from_dictionary', 'doubtful_unit',
       'standardized_value', 'factor', 'page_number_to_llm',
       'page_retrieval_scores', 'page_texts_to_llm', 'text_response_from_llm',
       'report_name_short', 'page_numbers_tried_by_llm',
       'automatic_extraction_tried', 'all_na', 'duplicate_flag', 'select_flag',
       '_merge'],
      dtype='object')

In [5]:
results_matched.head()

,Unnamed: 0,ReportName,automatic_extraction_tried,human_found_co2_emissions,scope_man,year_man,page_numbers_tried_by_llm,page_number_used_by_llm,page_retrieval_scores,page_man,...,val_name_man,type_man,ms_comment_man,extracted_scope_from_llm_orig,extracted_scope_from_llm,extracted_year_from_llm_orig,extracted_year_from_llm,page_texts_to_llm,text_response_from_llm,report_name
0,0,Allianz_2022_report.pdf,True,True,1,2013,"[48, 78, 79, 81, 91, 93, 133]",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Allianz_2022_report.pdf,True,True,1,2014,"[48, 78, 79, 81, 91, 93, 133]",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Allianz_2022_report.pdf,True,True,1,2015,"[48, 78, 79, 81, 91, 93, 133]",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Allianz_2022_report.pdf,True,True,1,2016,"[48, 78, 79, 81, 91, 93, 133]",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,Allianz_2022_report.pdf,True,True,1,2017,"[48, 78, 79, 81, 91, 93, 133]",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## How often is the true page the top ranked page?

We want to check whether the true page (from ground truth) is also the page with the highest retrieval score.

In [6]:
# For each report gather a list the pages the LLM tried with their page_retrieval_scores
results['page_numbers_tried_by_llm'] = results['page_numbers_tried_by_llm'].apply(tuple)
pages_infos = results[['report_name_short', 'page_number_used_by_llm', 'page_retrieval_scores', 'page_numbers_tried_by_llm']].copy()
pages_infos.drop_duplicates(inplace=True)

# Rank the pages per report by their retrieval score
pages_infos = pages_infos.sort_values(by=['report_name_short', 'page_retrieval_scores'], ascending=[True, False])
pages_infos['rank'] = pages_infos.groupby('report_name_short').cumcount() + 1

pages_infos


,report_name_short,page_number_used_by_llm,page_retrieval_scores,page_numbers_tried_by_llm,rank
18763,Allianz_2022_report.pdf,91,0.818319,"(48, 78, 79, 81, 91, 93, 133)",1
18631,Allianz_2022_report.pdf,48,0.817664,"(48, 78, 79, 81, 91, 93, 133)",2
18683,Allianz_2022_report.pdf,79,0.815491,"(48, 78, 79, 81, 91, 93, 133)",3
18671,Allianz_2022_report.pdf,78,0.814687,"(48, 78, 79, 81, 91, 93, 133)",4
18843,Allianz_2022_report.pdf,133,0.812585,"(48, 78, 79, 81, 91, 93, 133)",5
...,...,...,...,...,...
2392,xvivo perfusion_2021_report.pdf,91,0.745386,"(56, 73, 86, 88, 90, 91, 93)",3
2232,xvivo perfusion_2021_report.pdf,73,0.745279,"(56, 73, 86, 88, 90, 91, 93)",4
2192,xvivo perfusion_2021_report.pdf,56,0.742048,"(56, 73, 86, 88, 90, 91, 93)",5
2272,xvivo perfusion_2021_report.pdf,86,0.741537,"(56, 73, 86, 88, 90, 91, 93)",6


In [7]:
# Identify true pages from the matched results
true_pages = results_matched[['ReportName', 'page_man']].copy().drop_duplicates()

# Delete rows with NA in page_man
true_pages = true_pages[true_pages['page_man'].notna()]
true_pages



,ReportName,page_man
6,Allianz_2022_report.pdf,92
7,Allianz_2022_report.pdf,78
43,Daimler_2020_report.pdf,63
85,Fresenius SE_2019_report.pdf,126
129,acuity brands inc_2022_report.pdf,101
...,...,...
2765,varta ag_2021_report.pdf,39
2805,vital energy inc_2019_report.pdf,15
2818,vital energy inc_2019_report.pdf,66
2842,vital energy inc_2019_report.pdf,16


In [8]:
# Merge pages with ranks and page retrieval scores with true pages

true_pages_with_ranks = pd.merge(pages_infos, true_pages, left_on=['report_name_short', 'page_number_used_by_llm'], right_on=['ReportName', 'page_man'], how='outer', indicator=True)
true_pages_with_ranks

,report_name_short,page_number_used_by_llm,page_retrieval_scores,page_numbers_tried_by_llm,rank,ReportName,page_man,_merge
0,Allianz_2022_report.pdf,133,0.812585,"(48, 78, 79, 81, 91, 93, 133)",5.0,NaN,NaN,left_only
1,Allianz_2022_report.pdf,48,0.817664,"(48, 78, 79, 81, 91, 93, 133)",2.0,NaN,NaN,left_only
2,Allianz_2022_report.pdf,78,0.814687,"(48, 78, 79, 81, 91, 93, 133)",4.0,Allianz_2022_report.pdf,78,both
3,Allianz_2022_report.pdf,79,0.815491,"(48, 78, 79, 81, 91, 93, 133)",3.0,NaN,NaN,left_only
4,Allianz_2022_report.pdf,81,0.809103,"(48, 78, 79, 81, 91, 93, 133)",6.0,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...
972,xvivo perfusion_2021_report.pdf,86,0.741537,"(56, 73, 86, 88, 90, 91, 93)",6.0,NaN,NaN,left_only
973,xvivo perfusion_2021_report.pdf,88,0.737216,"(56, 73, 86, 88, 90, 91, 93)",7.0,NaN,NaN,left_only
974,xvivo perfusion_2021_report.pdf,90,0.746226,"(56, 73, 86, 88, 90, 91, 93)",2.0,NaN,NaN,left_only
975,xvivo perfusion_2021_report.pdf,91,0.745386,"(56, 73, 86, 88, 90, 91, 93)",3.0,NaN,NaN,left_only


In [26]:
true_pages_with_ranks[true_pages_with_ranks['_merge'] == 'both']['rank'].value_counts(normalize=True).sort_index()

rank
1.0    0.560976
2.0    0.195122
3.0    0.073171
4.0    0.085366
5.0    0.060976
6.0    0.024390
Name: proportion, dtype: float64

## How often is the true page the page with highest number of extractions (majority page)?

We want to identify the majority page of extractions for every report and compare it to the true page. 

In [10]:
# Identify the majority page used by the LLM for each report
not_na_results = results[results['extracted_value_from_llm'].notna()]
not_na_results.loc[:, 'majority_page'] = not_na_results.groupby('report_name_short')['page_number_used_by_llm'].transform(lambda x: x.mode()[0])

/var/folders/j6/3b1vrw2j3b98f0j_rm08ftgr0000gn/T/ipykernel_1165/3226106338.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  not_na_results.loc[:, 'majority_page'] = not_na_results.groupby('report_name_short')['page_number_used_by_llm'].transform(lambda x: x.mode()[0])


In [21]:
# Compare majority page with true page on row level
comparison = pd.merge(not_na_results,
                      true_pages, left_on=['report_name_short', 'page_number_used_by_llm'],
                      right_on=['ReportName', 'page_man'], how='outer')
comparison.loc[:, 'majority_is_true_page'] = comparison['majority_page'] == comparison['page_man']
comparison['majority_is_true_page'].value_counts(normalize=True)


majority_is_true_page
True     0.740964
False    0.259036
Name: proportion, dtype: float64

In [23]:
# Compare majority with true page on report level
comparison_page_level = pd.merge(not_na_results[['report_name_short', 'majority_page']].drop_duplicates(), 
                                 true_pages, 
                                 left_on = ['report_name_short'], 
                                 right_on = ['ReportName'])
comparison_page_level.loc[:, 'majority_is_true_page'] = comparison_page_level['majority_page'] == comparison_page_level['page_man']
comparison_page_level['majority_is_true_page'].value_counts(normalize=True)


majority_is_true_page
True     0.77381
False    0.22619
Name: proportion, dtype: float64

In [ ]:
# How often does the LLM extract from each rank (regardless of correctness)?
rank_lookup = pages_infos[['report_name_short', 'page_number_used_by_llm', 'rank']]

llm_rank_counts = results.merge(rank_lookup, on=['report_name_short', 'page_number_used_by_llm'], how='left')

# Keep only rows with a concrete extraction value
valid_llm_results = llm_rank_counts[
    llm_rank_counts['extracted_value_from_llm'].notna() &
    (llm_rank_counts['extracted_value_from_llm'] != "Not specified") &
    (llm_rank_counts['extracted_value_from_llm'] != "Nothing extracted. No Regex match")
]

valid_llm_results.shape[0]
valid_llm_results['report_name_short'].nunique()

rank_stats = (
    valid_llm_results[valid_llm_results['rank'].notna()]
    .assign(rank=lambda df: df['rank'].astype(int))
    .groupby('rank')['page_retrieval_scores']
    .agg(count='count', mean='mean', std='std', min='min', max='max')
)

rank_distribution = rank_stats.assign(share=lambda df: df['count'] / df['count'].sum())

rank_distribution

In [ ]:
# How often does the true page fall outside the retrieved set?
true_pages = results_matched[['ReportName', 'page_man']].copy().drop_duplicates()
true_pages = true_pages[true_pages['page_man'].notna()]

retrieved_pages_summary = (
    results[['report_name_short', 'page_numbers_tried_by_llm']]
    .drop_duplicates()
    .assign(page_numbers_tried_by_llm=lambda df: df['page_numbers_tried_by_llm'].apply(
        lambda pages: list(pages) if isinstance(pages, (list, tuple)) else []
    ))
    .groupby('report_name_short')['page_numbers_tried_by_llm']
    .apply(lambda page_lists: sorted({page for pages in page_lists for page in pages}))
    .rename('retrieved_pages')
    .reset_index()
)

truth_with_retrieval = true_pages.merge(
    retrieved_pages_summary,
    left_on='ReportName',
    right_on='report_name_short',
    how='left'
)

truth_with_retrieval['in_retrieved_set'] = truth_with_retrieval.apply(
    lambda row: row['page_man'] in row['retrieved_pages'] if isinstance(row['retrieved_pages'], list) else False,
    axis=1
)

retrieval_coverage = (
    truth_with_retrieval['in_retrieved_set']
    .value_counts(dropna=False)
    .rename('count')
    .to_frame()
    .assign(share=lambda df: df['count'] / df['count'].sum())
)

retrieval_coverage